## Education & Training Alignment

This section of the analysis focuses on how education and formal training align with the realities of the data science job market.  
We will explore three key research questions:

### Research Questions
1. **Education vs. Salary/Seniority**  
   - How does the required education level correlate with salary ranges and seniority level in data science roles?  
   - Example: Do master’s or PhD holders consistently earn more or secure higher-level positions compared to those with bachelor’s degrees or bootcamp training?

2. **Skills Gap in Formal Education**  
   - Are there technical skills or tools that are **highly demanded in job postings** but are **rarely taught** in formal education programs?  
   - Example: Cloud platforms (AWS, GCP, Azure), version control (Git/GitHub), or modern ML frameworks (TensorFlow, PyTorch).

3. **Job Postings vs. Curricula**  
   - How well do job postings align with the skills listed by educational institutions?  
   - Which **emerging skills** are appearing in postings but not yet common in curricula?  
   - Example: Generative AI tools, MLOps, or advanced data visualization libraries.

---

### Expected Outcomes
- Identify whether higher education is a strong predictor of salary and role seniority.  
- Highlight **gaps between industry demand and academic curricula**.  
- Provide recommendations for how training programs (bootcamps, universities, online courses) can better align with real-world employer expectations.

---

### Data Needed
- Education requirements from job postings.  
- Salary data and seniority levels (junior, mid, senior).  
- Curriculum data from universities, bootcamps, and online programs.  
- Frequency of technical skills mentioned in both postings and curricula.

In [25]:
import pandas as pd 
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

## Dataset Loading and Initial Exploration

We begin by loading the AI job dataset into a pandas DataFrame.  
To understand the dataset’s structure and quality, we perform three quick checks:  

1. **`info()`** – Displays the number of rows/columns, data types, and non-null counts.  
2. **`describe()`** – Summarizes numeric columns (count, mean, std, min, quartiles, max).  
3. **`head()`** – Previews the first 5 rows to confirm the data loaded correctly.  

These steps give us a foundational understanding of the dataset before deeper analysis.

In [26]:
# Load the AI job dataset from the datasets folder into a DataFrame called 'aijobs'
aijobs = pd.read_csv("../datasets/ai_job_dataset.csv")

# Display general information about the dataset:
# number of rows/columns, column names, data types, and non-null counts
aijobs.info()

# Generate basic descriptive statistics for numeric columns
# (count, mean, std, min, quartiles, max)
aijobs.describe()

# Preview the first 5 rows of the dataset
# This helps confirm the data loaded correctly and shows a sample of the values
aijobs.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   job_id                  15000 non-null  object 
 1   job_title               15000 non-null  object 
 2   salary_usd              15000 non-null  int64  
 3   salary_currency         15000 non-null  object 
 4   experience_level        15000 non-null  object 
 5   employment_type         15000 non-null  object 
 6   company_location        15000 non-null  object 
 7   company_size            15000 non-null  object 
 8   employee_residence      15000 non-null  object 
 9   remote_ratio            15000 non-null  int64  
 10  required_skills         15000 non-null  object 
 11  education_required      15000 non-null  object 
 12  years_experience        15000 non-null  int64  
 13  industry                15000 non-null  object 
 14  posting_date            15000 non-null

,job_id,job_title,salary_usd,salary_currency,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,education_required,years_experience,industry,posting_date,application_deadline,job_description_length,benefits_score,company_name
0,AI00001,AI Research Scientist,90376,USD,SE,CT,China,M,China,50,"Tableau, PyTorch, Kubernetes, Linux, NLP",Bachelor,9,Automotive,2024-10-18,2024-11-07,1076,5.9,Smart Analytics
1,AI00002,AI Software Engineer,61895,USD,EN,CT,Canada,M,Ireland,100,"Deep Learning, AWS, Mathematics, Python, Docker",Master,1,Media,2024-11-20,2025-01-11,1268,5.2,TechCorp Inc
2,AI00003,AI Specialist,152626,USD,MI,FL,Switzerland,L,South Korea,0,"Kubernetes, Deep Learning, Java, Hadoop, NLP",Associate,2,Education,2025-03-18,2025-04-07,1974,9.4,Autonomous Tech
3,AI00004,NLP Engineer,80215,USD,SE,FL,India,M,India,50,"Scala, SQL, Linux, Python",PhD,7,Consulting,2024-12-23,2025-02-24,1345,8.6,Future Systems
4,AI00005,AI Consultant,54624,EUR,EN,PT,France,S,Singapore,100,"MLOps, Java, Tableau, Python",Master,0,Media,2025-04-15,2025-06-23,1989,6.6,Advanced Robotics


### Check Dataset Dimensions

The `.shape` attribute tells us the size of the dataset in terms of rows and columns.  
This helps confirm the overall scale of the data we’re working with.  
Here, we also format the output to clearly state the row and column counts.

In [27]:
# Check the number of rows and columns in the dataset
# .shape[0] = number of rows, .shape[1] = number of columns
aijobs.shape
print(f"The dataset contains {aijobs.shape[0]:,} rows and {aijobs.shape[1]} columns.")


The dataset contains 15,000 rows and 19 columns.


### Check for Missing Values

Before analysis, it’s important to identify any missing data.  
The `.isnull().sum()` function shows the number of null values in each column.  
This helps us decide if we need to clean, impute, or drop certain columns/rows.

To ensure clean data for analysis, we remove all rows that contain any null values using `.dropna()`.  
We then confirm the cleanup by checking the total number of missing values again.  
If the result is `0`, it means the dataset has no remaining nulls.

In [28]:
# Check how many null (missing) values are present in each column
aijobs.isnull().sum()

job_id                    0
job_title                 0
salary_usd                0
salary_currency           0
experience_level          0
employment_type           0
company_location          0
company_size              0
employee_residence        0
remote_ratio              0
required_skills           0
education_required        0
years_experience          0
industry                  0
posting_date              0
application_deadline      0
job_description_length    0
benefits_score            0
company_name              0
dtype: int64

In [29]:
# Check the total number of missing values across the entire dataset
aijobs.isnull().sum().sum()

0

In [30]:
# Create a cleaned version of the dataset by dropping rows with missing values
aijobs_clean = aijobs.dropna()

# Verify that no missing values remain
aijobs_clean.isnull().sum().sum()

0

### Create Subset for Education & Training Alignment

For the education and training research questions, we only need a subset of the dataset.  
We filter the cleaned DataFrame to keep the most relevant columns:  

- `job_id` → unique job identifier  
- `salary_usd` → standardized salary measure  
- `experience_level` → seniority of the role  
- `education_required` → education level listed in posting  
- `years_experience` → required years of experience  
- `required_skills` → technical/soft skills required  
- `posting_date` → track emerging/temporal trends  
- `industry` → industry context for the role 

In [31]:
# Filter dataset to only include relevant columns for Education & Training Alignment analysis
education_jobs = aijobs_clean[["job_id", "salary_usd", "experience_level", "education_required","years_experience", "required_skills", "posting_date", "industry"]]
print(education_jobs)



        job_id  salary_usd experience_level education_required  \
0      AI00001       90376               SE           Bachelor   
1      AI00002       61895               EN             Master   
2      AI00003      152626               MI          Associate   
3      AI00004       80215               SE                PhD   
4      AI00005       54624               EN             Master   
...        ...         ...              ...                ...   
14995  AI14996       38604               EN           Bachelor   
14996  AI14997       57811               EN             Master   
14997  AI14998      189490               EX          Associate   
14998  AI14999       79461               EN                PhD   
14999  AI15000       56481               MI                PhD   

       years_experience                                  required_skills  \
0                     9         Tableau, PyTorch, Kubernetes, Linux, NLP   
1                     1  Deep Learning, AWS, Mathematic

In [32]:
# Save the subset to a new CSV file
education_jobs.to_csv('education_jobs.csv', index=False)

In [33]:
# --- Plotly + helpers ---
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ensure posting_date is datetime and added a Year-Month key
education_jobs = education_jobs.copy()
education_jobs['posting_date'] = pd.to_datetime(education_jobs['posting_date'], errors='coerce')
education_jobs['year_month']   = education_jobs['posting_date'].dt.to_period('M').astype(str)

# Normalize a few common education labels (optional/tweak if your data differs)
map_edu = {
    'Bachelors':'Bachelor', 'Masters':'Master', 'Ph.D':'PhD', 'Doctorate':'PhD',
    'Assoc':'Associate', 'Bootcamp':'Bootcamp', 'Certificate':'Certificate'
}
education_jobs['education_required'] = education_jobs['education_required'].replace(map_edu)

# Parse required_skills into lists
def split_skills(s):
    if pd.isna(s): 
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

education_jobs['skills_list'] = education_jobs['required_skills'].apply(split_skills)


In [34]:
# --- Boxplot: Salary distribution by education level ---
fig1 = px.box(
    education_jobs,
    x='education_required',
    y='salary_usd',
    points='suspectedoutliers',
    title='Salary Distribution by Education Level',
    labels={'education_required':'Education Level','salary_usd':'Salary (USD)'},
)
fig1.update_layout(xaxis={'categoryorder':'array','categoryarray':sorted(education_jobs['education_required'].unique())})
fig1.show()

# --- Bar: Average salary by education level ---
avg_salary_edu = (education_jobs
                  .groupby('education_required', as_index=False)['salary_usd']
                  .mean()
                  .sort_values('salary_usd', ascending=False))
fig2 = px.bar(
    avg_salary_edu,
    x='education_required', y='salary_usd',
    title='Average Salary by Education Level',
    labels={'education_required':'Education Level','salary_usd':'Avg Salary (USD)'},
    text_auto='.0f'
)
fig2.update_traces(textposition='outside')
fig2.update_layout(xaxis={'categoryorder':'array','categoryarray':avg_salary_edu['education_required']})
fig2.show()

# --- Stacked distribution: Education vs Experience Level ---
edu_exp = (education_jobs
           .groupby(['education_required','experience_level'])
           .size()
           .reset_index(name='count'))
fig3 = px.bar(
    edu_exp,
    x='education_required', y='count',
    color='experience_level',
    title='Role Seniority by Education Level',
    labels={'education_required':'Education Level','count':'Number of Roles','experience_level':'Seniority'},
    barmode='stack'
)
fig3.update_layout(xaxis={'categoryorder':'array','categoryarray':sorted(education_jobs['education_required'].unique())})
fig3.show()


Question 1 Answer: The analysis shows that education level has a limited influence on salary or seniority in data science roles. Across Associate, Bachelor, Master, and PhD degrees, median salaries cluster around $100K, and average salaries differ by only a few thousand dollars. The boxplot and bar chart reveal significant overlap in salary ranges, with high earners present at every education level. Similarly, the seniority distribution shows that professionals with Bachelor’s and Master’s degrees are just as likely to hold mid-, senior-, or executive-level positions as those with PhDs. Overall, the data suggests that while higher education can provide a foundation or early-career advantage, long-term growth and compensation in data science are driven primarily by experience, technical expertise, and applied skills rather than academic credentials.

In [35]:
# Top N skills in job postings
from collections import Counter

skill_counts = Counter([skill for row in education_jobs['skills_list'] for skill in row])
skills_df = (pd.DataFrame(skill_counts.items(), columns=['skill','frequency'])
             .sort_values('frequency', ascending=False)
             .reset_index(drop=True))

TOP_N = 20
top_skills = skills_df.head(TOP_N)

fig4 = px.bar(
    top_skills.sort_values('frequency'),
    x='frequency', y='skill',
    orientation='h',
    title=f'Top {TOP_N} Skills in Job Postings',
    labels={'frequency':'Frequency','skill':'Skill'},
    text='frequency'
)
fig4.update_traces(textposition='outside')
fig4.show()


Question 2 Answer: The analysis of the top twenty skills in job postings reveals that SQL, Kubernetes, PyTorch, Git, and Google Cloud Platform (GCP) are among the most in-demand tools in data science roles. The high frequency of these skills highlights the industry’s strong emphasis on data infrastructure, cloud computing, and machine learning frameworks. Interestingly, many of these technologies—such as Kubernetes, GCP, and MLOps-related tools—are not traditionally emphasized in formal education programs, suggesting a growing skills gap between academic training and workplace requirements. Employers appear to prioritize candidates who can apply modern, production-ready technologies rather than those with only theoretical knowledge. This finding underscores the need for curricula and training programs to expand coverage of cloud platforms, version control, and machine learning deployment skills to better align with real-world job market demands.

In [36]:
# Skills to track over time
skills_to_track = ['Python', 'TensorFlow', 'PyTorch', 'MLOps', 'AWS', 'GCP', 'Azure', 'Git', 'Generative AI']

# Build a long table of skill-month counts
records = []
for sk in skills_to_track:
    mask = education_jobs['required_skills'].str.contains(fr'\b{sk}\b', case=False, na=False, regex=True)
    tmp = (education_jobs[mask]
           .groupby('year_month')
           .size()
           .reset_index(name='count'))
    tmp['skill'] = sk
    records.append(tmp)

trend_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame(columns=['year_month','count','skill'])

# Interactive line chart with facet by skill
fig5 = px.line(
    trend_df,
    x='year_month', y='count', color='skill',
    markers=True,
    title='Emerging Skills in Job Postings Over Time',
    labels={'year_month':'Year-Month','count':'Postings Mentioning Skill','skill':'Skill'}
)
fig5.update_xaxes(type='category', tickangle=45)
fig5.show()


Question 3 Answer: The trend analysis of emerging skills in job postings shows that Python remains the most consistently mentioned skill, maintaining steady demand throughout the observed period. However, newer and more specialized tools such as TensorFlow, PyTorch, MLOps, and cloud platforms (AWS, GCP, Azure) are steadily rising in frequency, reflecting the industry’s ongoing shift toward AI deployment, automation, and scalable machine learning systems. The increasing mentions of MLOps and cloud technologies suggest that employers now prioritize end-to-end data science capabilities — not just modeling, but also operationalization and integration into production environments. This pattern indicates that while foundational skills like Python remain essential, curricula and training programs must evolve to include Generative AI frameworks, MLOps practices, and cloud-based machine learning to stay aligned with market trends and employer expectations.

In [37]:
# Encode categories for a quick correlation check
encoded = education_jobs.copy()
encoded['edu_code'] = encoded['education_required'].astype('category').cat.codes
encoded['exp_code'] = encoded['experience_level'].astype('category').cat.codes

corr_df = encoded[['salary_usd','years_experience','edu_code','exp_code']].corr().round(2)

fig6 = px.imshow(
    corr_df,
    text_auto=True,
    title='Correlation Between Salary, Experience, and Encoded Categories',
    labels=dict(color='Correlation')
)
fig6.show()


The correlation matrix confirms that years of experience has the strongest relationship with salary (r = 0.74), indicating that experience is a major driver of compensation in data science roles. In contrast, the education level (edu_code) shows virtually no correlation (r ≈ 0.00) with salary, reinforcing earlier findings that academic credentials have limited influence on pay outcomes. The experience level (exp_code) also has a weak correlation with salary (r = 0.14), suggesting that formal job titles or levels contribute less to salary variance than total years of professional experience. Overall, this quantitative view supports the conclusion that experience and applied expertise—not educational attainment—are the most significant predictors of salary growth in data science careers.

In [38]:
# Save skill frequencies
skills_df.to_csv('skill_frequencies.csv', index=False)

# Salary by education level stats
salary_stats = (education_jobs
                .groupby('education_required')['salary_usd']
                .agg(['count','mean','median','min','max'])
                .reset_index()
                .sort_values('median', ascending=False))
salary_stats.to_csv('salary_by_education_stats.csv', index=False)

# Seniority-by-education distribution
edu_exp.to_csv('education_vs_experience_counts.csv', index=False)

# Trend_df above:
trend_df.to_csv('skill_trends_over_time.csv', index=False)


In [39]:
# Interactive Skill Dashboard (ipywidgets)
import re
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import Dropdown, Output, VBox
from IPython.display import display, clear_output

# Build the universe of skills from parsed list 
all_skills = sorted({s for row in education_jobs['skills_list'] for s in row})
default_skill = 'Python' if 'Python' in all_skills else (all_skills[0] if all_skills else None)

# Compute trend for one skill
def skill_trend(skill: str):
    if not skill:
        return education_jobs.assign(year_month=education_jobs['posting_date'].dt.to_period('M').astype(str)) \
                             .groupby('year_month').size().reset_index(name='count').assign(skill='(none)')
    # word-boundary match to avoid partials (e.g., "Git" vs "Gita")
    pat = rf'(?<!\w){re.escape(skill)}(?!\w)'
    mask = education_jobs['required_skills'].str.contains(pat, case=False, na=False, regex=True)
    df = (education_jobs[mask]
          .groupby('year_month')
          .size()
          .reset_index(name='count')
          .sort_values('year_month', key=lambda s: s.astype(str)))
    df['skill'] = skill
    return df

# UI + rendering
out = Output()

def render(skill):
    df = skill_trend(skill)
    title = f'Postings Mentioning "{skill}" Over Time' if skill else 'Postings Over Time'
    fig_line = px.line(
        df, x='year_month', y='count', markers=True,
        title=title, labels={'year_month':'Year-Month', 'count':'Count'}
    )
    fig_line.update_xaxes(type='category', tickangle=45)

    # Table (sorted desc by date for quick scanning)
    df_tbl = df.sort_values('year_month', ascending=False)
    fig_tbl = go.Figure(data=[go.Table(
        header=dict(values=['Year-Month','Count'], fill_color='#f2f2f2', align='left'),
        cells=dict(values=[df_tbl['year_month'], df_tbl['count']], align='left')
    )])
    fig_tbl.update_layout(title=f'Data Table • {skill}')

    with out:
        clear_output(wait=True)
        fig_line.show()
        fig_tbl.show()

dropdown = Dropdown(options=all_skills, value=default_skill, description='Skill:')
def on_change(change):
    if change['name'] == 'value' and change['new'] != change['old']:
        render(change['new'])
dropdown.observe(on_change, names='value')

display(VBox([dropdown, out]))
render(default_skill)


# Summary: Education & Training Alignment Analysis

### Purpose
This analysis explored how education, training, and technical skills align with what employers are actually looking for in today’s data science job market. I wanted to see whether higher education truly impacts salary or seniority, identify which skills are most in-demand, and understand where education programs might be falling behind industry needs.

---

### Key Findings

#### 1. Education vs. Salary & Seniority
Across Associate, Bachelor, Master, and PhD degrees, salaries were almost identical — with median pay sitting near **$100K**. The small salary differences between degrees show that **education level alone doesn’t guarantee higher earnings**. Even more interesting, seniority levels were distributed evenly across education types, meaning **career advancement happens through experience and skill growth**, not necessarily higher degrees. This was backed up by the correlation matrix — **years of experience had the strongest connection to salary (r = 0.74)**, while education had almost none.  

**Takeaway:**  
Your degree might help you get in the door, but it’s **real-world experience and technical depth** that drive long-term growth and pay increases.

---

#### 2. Skills Gap in Formal Education
The top skills across postings were **SQL, Kubernetes, PyTorch, Git, and GCP**, with other frequent mentions like **Tableau, Deep Learning, and Azure**. These are hands-on, production-level tools that many traditional programs don’t cover deeply. It’s clear that **the industry is prioritizing practical, deployable skills** over theory — especially cloud platforms, MLOps, and machine learning engineering tools.  

**Takeaway:**  
There’s a growing **skills gap** between what’s taught and what’s expected.  
Training programs should spend more time on **cloud computing, version control, and model deployment**, not just classroom analytics.

---

#### 3. Job Postings vs. Curricula (Emerging Skills)
Over time, tools like **MLOps, TensorFlow, PyTorch, and cloud technologies (AWS, GCP, Azure)** have become more common in job postings. **Python** is still foundational, but it’s no longer the thing that sets candidates apart — employers now want people who can take a model from notebook to production.  

**Takeaway:**  
The field is shifting toward **AI deployment and automation**, and that needs to be reflected in how future data scientists are trained.

---

### Overall Reflection
After analyzing the data, it’s clear that **education alone doesn’t determine success in data science**. Employers value candidates who can combine technical expertise with real-world application — people who don’t just know the tools, but know how to use them. For students and professionals, this means focusing on **building projects, earning certifications, and staying adaptable** as tools and technologies evolve.  

---

### Moving Forward
- **For Educators:** Bring more real-world projects into the curriculum and teach skills that connect to industry tools.  
- **For Learners:** Keep sharpening practical skills — focus on cloud platforms, Git, MLOps, and advanced AI workflows.  
- **For Employers:** Partner with education programs to align expectations and help bridge the gap between learning and hiring.

---

### Final Thought
The takeaway from this analysis is simple:  **Degrees open doors, but skills keep them open.**  
Success in data science depends on staying curious, practicing continuously, and building the kind of hands-on experience that employers can see, trust, and value.
